In [1]:
!pip install tqdm
# ==========================================
# STEP 1 & 2: LOAD DATA & CLEAN TEXT
# ==========================================

import logging
import pandas as pd
import re
import spacy
from pathlib import Path
from typing import Optional
from tqdm import tqdm  # For progress bars

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('resume_processing.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Configuration
DATA_PATH = Path('Resume.csv')
USE_HTML_FALLBACK = True  # Use HTML column if text column is empty
BATCH_SIZE = 100  # Process in batches for memory efficiency

class ResumeCleaner:
    """
    Resume text cleaner with spaCy NLP pipeline.
    Handles batch processing and fallback mechanisms.
    """
    
    def __init__(self, spacy_model: str = 'en_core_web_sm'):
        """
        Initialize the cleaner with a spaCy model.
        Loads model once at initialization, not per row.
        """
        logger.info(f"Loading spaCy model: {spacy_model}")
        self.nlp = spacy.load(spacy_model)
        logger.info("spaCy model loaded successfully")
        
        # Compile regex patterns once for efficiency
        self.url_pattern = re.compile(r'http\S+')
        self.email_pattern = re.compile(r'\S+@\S+')
        self.special_chars_pattern = re.compile(r'[^a-z0-9\s]')
        self.newline_pattern = re.compile(r'\n')
        self.nbsp_pattern = re.compile(r'\xa0')
        self.whitespace_pattern = re.compile(r'\s+')
        
    def clean_single_resume(self, text: str) -> str:
        """
        Clean a single resume text.
        
        Args:
            text: Raw resume text
            
        Returns:
            Cleaned and lemmatized text
        """
        if not isinstance(text, str) or pd.isna(text) or text.strip() == '':
            return ""
        
        # Step 1: Lowercase
        text = text.lower()
        
        # Step 2: Remove URLs and emails
        text = self.url_pattern.sub(' ', text)
        text = self.email_pattern.sub(' ', text)
        
        # Step 3: Remove special characters (preserve spaces)
        text = self.special_chars_pattern.sub(' ', text)
        
        # Step 4: Normalize whitespace and line breaks
        text = self.newline_pattern.sub(' ', text)
        text = self.nbsp_pattern.sub(' ', text)
        text = self.whitespace_pattern.sub(' ', text).strip()
        
        # Skip processing if text is too short
        if len(text) < 10:
            return ""
        
        # Step 5: Process with spaCy (lemmatization + stopword removal)
        # Note: Process stopwords BEFORE lemmatization for better context
        doc = self.nlp(text)
        
        # Filter: not stopword, not punctuation, length > 2, and not purely numeric
        cleaned_tokens = [
            token.lemma_ for token in doc 
            if not token.is_stop 
            and not token.is_punct 
            and len(token.text) > 2 
            and not token.text.isdigit()
        ]
        
        return " ".join(cleaned_tokens)
    
    def clean_batch(self, texts: pd.Series, use_html_fallback: bool = True, html_column: Optional[pd.Series] = None) -> list:
        """
        Clean a batch of resumes with progress tracking.
        
        Args:
            texts: Series of resume text
            use_html_fallback: Whether to use HTML column as fallback
            html_column: Series of HTML text (for fallback)
            
        Returns:
            List of cleaned texts
        """
        cleaned_texts = []
        
        # Process with tqdm progress bar
        for idx, text in tqdm(texts.items(), total=len(texts), desc="Cleaning resumes"):
            cleaned = self.clean_single_resume(text)
            
            # Fallback to HTML if text is empty and HTML is available
            if use_html_fallback and html_column is not None and not cleaned:
                html_text = html_column[idx]
                if isinstance(html_text, str):
                    # Simple HTML to text conversion (strip tags)
                    html_text = re.sub(r'<[^>]+>', ' ', html_text)
                    cleaned = self.clean_single_resume(html_text)
            
            cleaned_texts.append(cleaned)
        
        # Log statistics
        empty_count = sum(1 for t in cleaned_texts if not t)
        logger.info(f"Cleaned {len(cleaned_texts)} resumes. {empty_count} empty after cleaning.")
        
        return cleaned_texts


def load_and_validate_data(filepath: Path) -> pd.DataFrame:
    """
    Load and validate the resume dataset.
    
    Args:
        filepath: Path to CSV file
        
    Returns:
        Validated DataFrame
    """
    logger.info(f"Loading data from: {filepath}")
    
    if not filepath.exists():
        raise FileNotFoundError(f"Data file not found: {filepath}")
    
    try:
        df = pd.read_csv(filepath)
        logger.info(f"Loaded {df.shape[0]} resumes with {df.shape[1]} columns")
    except Exception as e:
        logger.error(f"Failed to load CSV: {e}")
        raise
    
    # Validate required columns
    required_cols = ['ID', 'Resume_str', 'Category']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Check for missing values
    missing_resumes = df['Resume_str'].isna().sum()
    if missing_resumes > 0:
        logger.warning(f"Found {missing_resumes} resumes with missing text. Will attempt fallback to HTML.")
        df['Resume_str'] = df['Resume_str'].fillna('')
    
    # Optional: Drop HTML column to save memory (but keep if needed for fallback)
    html_available = 'Resume_html' in df.columns
    
    if not html_available and USE_HTML_FALLBACK:
        logger.warning("HTML column not found. Fallback will be disabled.")
    
    return df, html_available


def main():
    """Main execution function."""
    logger.info("=" * 50)
    logger.info("Resume Screening System - Data Preprocessing")
    logger.info("=" * 50)
    
    # 1. Load and validate data
    try:
        df, has_html = load_and_validate_data(DATA_PATH)
    except Exception as e:
        logger.error(f"Data loading failed: {e}")
        return
    
    # 2. Initialize cleaner
    cleaner = ResumeCleaner()
    
    # 3. Clean resumes
    html_column = df['Resume_html'] if has_html else None
    
    cleaned_resumes = cleaner.clean_batch(
        texts=df['Resume_str'],
        use_html_fallback=USE_HTML_FALLBACK,
        html_column=html_column
    )
    
    # 4. Add cleaned column
    df['Cleaned_Resume'] = cleaned_resumes
    
    # 5. Remove empty resumes (optional - log warning)
    empty_mask = df['Cleaned_Resume'] == ''
    if empty_mask.any():
        logger.warning(f"Removing {empty_mask.sum()} resumes with empty text after cleaning")
        df = df[~empty_mask].reset_index(drop=True)
    
    # 6. Preview results
    logger.info("✅ Text Cleaning Complete!")
    logger.info(f"Final dataset: {df.shape[0]} resumes")
    
    # Display sample
    sample_cols = ['Category', 'Resume_str', 'Cleaned_Resume']
    sample = df[sample_cols].head(2)
    
    print("\n" + "=" * 60)
    print("SAMPLE RESULTS")
    print("=" * 60)
    for idx, row in sample.iterrows():
        print(f"\n--- Resume {idx+1} (Category: {row['Category']}) ---")
        print(f"Original (first 200 chars): {row['Resume_str'][:200]}...")
        print(f"Cleaned (first 200 chars):   {row['Cleaned_Resume'][:200]}...")
    
    return df


if __name__ == "__main__":
    df = main()


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
2026-04-01 23:26:30,134 - INFO - ==================================================
2026-04-01 23:26:30,137 - INFO - Resume Screening System - Data Preprocessing
2026-04-01 23:26:30,140 - INFO - ==================================================
2026-04-01 23:26:30,142 - INFO - Loading data from: Resume.csv
2026-04-01 23:26:32,181 - INFO - Loaded 2484 resumes with 4 columns
2026-04-01 23:26:32,195 - INFO - Loading spaCy model: en_core_web_sm
2026-04-01 23:26:33,911 - INFO - spaCy model loaded successfully
Cleaning resumes: 100%|████████████████████████████████████████████████████████████| 2484/2484 [12:25<00:00,  3.33it/s]
2026-04-01 23:38:59,948 - INFO - Cleaned 2484 resumes. 1 empty after cleaning.
2026-04-01 23:39:00,196 - WARNING - Removing 1 resumes with empty text after cleaning
--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\akuma


SAMPLE RESULTS

--- Resume 1 (Category: HR) ---
Original (first 200 chars):          HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Resp...
Cleaned (first 200 chars):   administrator marketing associate administrator summary dedicated customer service manager year experience hospitality customer service management respect builder leader customer focus team strive ins...

--- Resume 2 (Category: HR) ---
Original (first 200 chars):          HR SPECIALIST, US HR OPERATIONS       Summary     Versatile  media professional with background in Communications, Marketing, Human Resources and Technology.         Experience     09/2015   ...
Cleaned (first 200 chars):   specialist operation summary versatile media professional background communication marketing human resource technology experience current specialist operation company city state manage communi

In [2]:
# ==========================================
# STEP 3: ADVANCED ATS MATCHING ENGINE
# ==========================================

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Union
from dataclasses import dataclass, field
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
import logging
import json
from collections import Counter
import re

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


@dataclass
class MatchResult:
    """Data class to store individual match results with skill analysis."""
    candidate_id: Union[int, str]
    category: str
    raw_score: float
    normalized_score: float
    rank: int
    missing_skills: List[str] = field(default_factory=list)
    matched_skills: List[str] = field(default_factory=list)
    
    def to_dict(self) -> dict:
        return {
            'candidate_id': self.candidate_id,
            'category': self.category,
            'match_score': f"{self.normalized_score:.1f}%",
            'rank': self.rank,
            'missing_skills': self.missing_skills[:5],  
            'matched_skills': self.matched_skills[:10]  
        }


class SkillExtractor:
    """
    Domain-specific skill extraction for HR/Technical roles.
    Uses pattern matching + configurable skill dictionaries.
    """
    def __init__(self, custom_skills: Optional[Dict[str, List[str]]] = None):
        self.skill_dict = {
            # Programming Languages
            'python': ['python', 'py', 'python3', 'python 3'],
            'java': ['java', 'j2ee', 'jdk'],
            'javascript': ['javascript', 'js', 'node.js', 'nodejs', 'react', 'angular', 'vue'],
            'c++': ['c++', 'cpp', 'c plus plus'],
            'sql': ['sql', 'mysql', 'postgresql', 'postgres', 'database', 'rdbms'],
            
            # ML/AI
            'machine learning': ['machine learning', 'ml', 'predictive modeling', 'predictive model'],
            'deep learning': ['deep learning', 'dl', 'neural network', 'tensorflow', 'pytorch', 'keras'],
            'nlp': ['nlp', 'natural language processing', 'text mining'],
            'computer vision': ['computer vision', 'cv', 'image processing', 'opencv'],
            
            # Data Science
            'data science': ['data science', 'data scientist', 'analytics'],
            'data analysis': ['data analysis', 'exploratory data analysis', 'eda'],
            'statistics': ['statistics', 'statistical analysis', 'hypothesis testing'],
            'pandas': ['pandas', 'dataframe'],
            'numpy': ['numpy', 'numerical'],
            'scikit-learn': ['scikit-learn', 'sklearn'],
            
            # Cloud & Tools
            'aws': ['aws', 'amazon web services', 'ec2', 's3'],
            'docker': ['docker', 'container', 'kubernetes', 'k8s'],
            'git': ['git', 'github', 'gitlab', 'version control'],
            
            # HR-Specific Skills (Added for this dataset)
            'employee relations': ['employee relations', 'labor relations', 'union relations'],
            'recruitment': ['recruitment', 'recruiting', 'talent acquisition', 'staffing', 'hiring'],
            'benefits administration': ['benefits', 'benefits administration', 'compensation', '401k'],
            'hr compliance': ['compliance', 'fmla', 'ada', 'eeo', 'labor law', 'employment law'],
            'performance management': ['performance management', 'performance review', 'appraisal'],
            'training': ['training', 'development', 'onboarding', 'orientation'],
            'hr generalist': ['hr generalist', 'human resources generalist'],
            'payroll': ['payroll', 'payroll processing', 'payroll administration'],
        }
        
        if custom_skills:
            self.skill_dict.update(custom_skills)
            
        self.skill_patterns = {}
        for canonical_skill, synonyms in self.skill_dict.items():
            pattern = r'\b(' + '|'.join(re.escape(s) for s in synonyms) + r')\b'
            self.skill_patterns[canonical_skill] = re.compile(pattern, re.IGNORECASE)
            
    def extract_skills(self, text: str) -> List[str]:
        if not text or not isinstance(text, str):
            return []
        found_skills = set()
        for canonical_skill, pattern in self.skill_patterns.items():
            if pattern.search(text):
                found_skills.add(canonical_skill)
        return sorted(list(found_skills))
        
    def calculate_skill_overlap(self, resume_skills: List[str], job_skills: List[str]) -> Tuple[List[str], List[str], float]:
        resume_set = set(resume_skills)
        job_set = set(job_skills)
        matched = list(resume_set.intersection(job_set))
        missing = list(job_set.difference(resume_set))
        
        if not job_set:
            match_pct = 0.0
        else:
            match_pct = len(matched) / len(job_set) * 100
        return matched, missing, match_pct


class ATSMatcher:
    """
    Advanced Matching Engine for Applicant Tracking System.
    Combines TF-IDF similarity with skill-based matching for better results.
    """
    def __init__(
        self,
        max_features: int = 10000,
        ngram_range: Tuple[int, int] = (1, 3), 
        skill_weight: float = 0.3, 
        use_category_filter: bool = True,
        model_dir: Optional[Path] = None
    ):
        self.vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=ngram_range,
            stop_words='english',
            min_df=2, 
            max_df=0.95 
        )
        self.skill_extractor = SkillExtractor()
        self.resume_vectors = None
        self.resume_skills = None 
        self.category_mapping = None
        self.is_fitted = False
        self.skill_weight = skill_weight
        self.use_category_filter = use_category_filter
        self.model_dir = model_dir or Path('models')
        self.model_dir.mkdir(exist_ok=True, parents=True)
        self.score_scaler = MinMaxScaler()
        
    def _extract_batch_skills(self, texts: pd.Series) -> List[List[str]]:
        """Extract skills from a batch of texts with progress tracking."""
        skills_list = []
        from tqdm import tqdm
        for text in tqdm(texts, desc="Extracting skills", leave=False):
            skills = self.skill_extractor.extract_skills(text)
            skills_list.append(skills)
        return skills_list
        
    def _build_category_mapping(self, categories: pd.Series) -> Dict[str, List[int]]:
        """Build mapping from category to indices for filtering."""
        mapping = {}
        for idx, cat in enumerate(categories):
            if cat not in mapping:
                mapping[cat] = []
            mapping[cat].append(idx)
        return mapping
        
    def fit(self, resumes: pd.Series, categories: Optional[pd.Series] = None):
        """Train the matcher on a corpus of resumes."""
        logger.info("=" * 50)
        logger.info("Training ATS Matcher")
        logger.info("=" * 50)
        
        # Validate input
        if resumes is None or len(resumes) == 0:
            raise ValueError("Resumes series cannot be empty")
        
        # Filter out empty strings to avoid warnings
        resumes = resumes.fillna('')
        non_empty_mask = resumes.str.len() > 0
        if not non_empty_mask.all():
            logger.warning(f"Found {(~non_empty_mask).sum()} empty resumes. They will be included but may produce zero vectors.")
        
        logger.info(f"Converting {len(resumes)} resumes to TF-IDF vectors...")
        try:
            self.resume_vectors = self.vectorizer.fit_transform(resumes)
            logger.info(f"Vocabulary size: {self.resume_vectors.shape[1]} terms")
        except Exception as e:
            logger.error(f"TF-IDF transformation failed: {e}")
            raise
        
        logger.info("Extracting skills from resumes...")
        self.resume_skills = self._extract_batch_skills(resumes)
        
        if categories is not None and self.use_category_filter:
            self.category_mapping = self._build_category_mapping(categories)
            logger.info(f"Built category mapping for {len(self.category_mapping)} categories")
            
        self.is_fitted = True
        logger.info("✅ Training complete!")
    
    def save_model(self, name: str = "ats_matcher"):
        """Save the trained model components to disk."""
        if not self.is_fitted:
            logger.warning("Model not fitted. Nothing to save.")
            return
        
        model_path = self.model_dir / f"{name}.joblib"
        vectorizer_path = self.model_dir / f"{name}_vectorizer.joblib"
        
        try:
            joblib.dump(self.vectorizer, vectorizer_path)
            
            model_data = {
                'resume_vectors': self.resume_vectors,
                'resume_skills': self.resume_skills,
                'category_mapping': self.category_mapping,
                'is_fitted': self.is_fitted,
                'skill_weight': self.skill_weight,
                'use_category_filter': self.use_category_filter
            }
            joblib.dump(model_data, model_path)
            logger.info(f"Model saved to {model_path} and {vectorizer_path}")
        except Exception as e:
            logger.error(f"Failed to save model: {e}")
            raise
    
    def load_model(self, name: str = "ats_matcher"):
        """Load trained model components from disk."""
        model_path = self.model_dir / f"{name}.joblib"
        vectorizer_path = self.model_dir / f"{name}_vectorizer.joblib"
        
        if not model_path.exists():
            raise FileNotFoundError(f"Model file not found: {model_path}")
        if not vectorizer_path.exists():
            raise FileNotFoundError(f"Vectorizer file not found: {vectorizer_path}")
        
        try:
            self.vectorizer = joblib.load(vectorizer_path)
            model_data = joblib.load(model_path)
            
            self.resume_vectors = model_data['resume_vectors']
            self.resume_skills = model_data['resume_skills']
            self.category_mapping = model_data['category_mapping']
            self.is_fitted = model_data['is_fitted']
            self.skill_weight = model_data['skill_weight']
            self.use_category_filter = model_data['use_category_filter']
            
            logger.info(f"Model loaded from {model_path}")
        except Exception as e:
            logger.error(f"Failed to load model: {e}")
            raise
        
    def rank_candidates(
        self,
        job_description: str,
        df_resumes: pd.DataFrame,
        job_category: Optional[str] = None,
        top_n: int = 10,
        min_score_threshold: float = 20.0
    ) -> pd.DataFrame:
        """Rank candidates against a job description."""
        if not self.is_fitted:
            raise ValueError("ATSMatcher must be fitted with resumes before ranking.")
        
        if not job_description or not isinstance(job_description, str):
            raise ValueError("Job description must be a non-empty string.")
        
        if df_resumes is None or len(df_resumes) == 0:
            raise ValueError("Resumes DataFrame cannot be empty.")
            
        jd_cleaned = self._clean_job_description(job_description)
        jd_skills = self.skill_extractor.extract_skills(jd_cleaned)
        logger.info(f"Extracted {len(jd_skills)} skills from job description: {jd_skills[:10]}")
        
        indices_to_consider = None
        if job_category and self.category_mapping and job_category in self.category_mapping:
            indices_to_consider = self.category_mapping[job_category]
            logger.info(f"Filtering to {len(indices_to_consider)} candidates in category: {job_category}")
            
        jd_vector = self.vectorizer.transform([jd_cleaned])
        
        if indices_to_consider is not None:
            filtered_vectors = self.resume_vectors[indices_to_consider]
            similarities = cosine_similarity(jd_vector, filtered_vectors).flatten()
            full_similarities = np.zeros(len(df_resumes))
            for i, orig_idx in enumerate(indices_to_consider):
                full_similarities[orig_idx] = similarities[i]
        else:
            full_similarities = cosine_similarity(jd_vector, self.resume_vectors).flatten()
            
        skill_scores = np.zeros(len(df_resumes))
        skill_gaps = {}
        
        for idx, resume_skills in enumerate(self.resume_skills):
            matched, missing, match_pct = self.skill_extractor.calculate_skill_overlap(
                resume_skills, jd_skills
            )
            skill_scores[idx] = match_pct / 100.0 
            skill_gaps[idx] = {
                'matched': matched,
                'missing': missing,
                'pct': match_pct
            }
            
        combined_scores = (1 - self.skill_weight) * full_similarities + self.skill_weight * skill_scores
        
        results_df = df_resumes.copy()
        results_df['Similarity_Score'] = np.round(full_similarities * 100, 2)
        results_df['Skill_Score'] = np.round(skill_scores * 100, 2)
        results_df['Combined_Score'] = np.round(combined_scores * 100, 2)
        
        results_df['Matched_Skills'] = [', '.join(skill_gaps[i]['matched'][:10]) for i in range(len(results_df))]
        results_df['Missing_Skills'] = [', '.join(skill_gaps[i]['missing'][:5]) for i in range(len(results_df))]
        
        results_df = results_df[results_df['Combined_Score'] >= min_score_threshold]
        results_df = results_df.sort_values(by='Combined_Score', ascending=False).reset_index(drop=True)
        results_df['Rank'] = range(1, len(results_df) + 1)
        
        logger.info(f"Found {len(results_df)} candidates above {min_score_threshold}% threshold")
        return results_df.head(top_n)
        
    def _clean_job_description(self, text: str) -> str:
        """Clean job description text for consistency with resume cleaning."""
        if not text:
            return ""
        text = text.lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text


def display_results(results_df: pd.DataFrame, show_details: bool = True):
    """Display ranking results in a recruiter-friendly format."""
    if results_df is None or len(results_df) == 0:
        print("\n⚠️ No results to display.")
        return
        
    print("\n" + "="*80)
    print("🏆 TOP CANDIDATE RANKINGS 🏆")
    print("="*80)
    
    display_cols = ['Rank', 'Category', 'Combined_Score', 'Similarity_Score', 'Skill_Score']
    if show_details:
        display_cols.extend(['Matched_Skills', 'Missing_Skills'])
        
    display_df = results_df[display_cols].copy()
    for col in ['Combined_Score', 'Similarity_Score', 'Skill_Score']:
        if col in display_df.columns:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.1f}%")
            
    print(display_df.to_string(index=False))
    print("="*80)


# ==========================================
# TEST THE ENHANCED ENGINE (CORRECTED)
# ==========================================

if __name__ == "__main__":
    
    # OPTION 1: Data Scientist/Technical Role
    DS_JOB_DESCRIPTION = """
    We are looking for a Data Scientist or Python Developer. 
    The ideal candidate must have strong skills in Python, Machine Learning, 
    data analysis, pandas, and scikit-learn. Experience with artificial intelligence, 
    statistics, and data visualization is highly preferred.
    Strong communication and problem-solving skills are required.
    """
    
    # OPTION 2: HR Role 
    HR_JOB_DESCRIPTION = """
    We are looking for an experienced Human Resources professional.
    The ideal candidate must have expertise in:
    - Employee Relations and conflict resolution
    - Recruitment and talent acquisition
    - Benefits administration and compensation
    - HR policies and compliance (FMLA, ADA, EEO)
    - Performance management and employee development
    - HRIS systems and payroll processing
    Strong communication, leadership, and organizational skills are essential.
    """
    
    logger.info("Initializing Enhanced ATS Matcher...")
    matcher = ATSMatcher(
        max_features=8000,
        ngram_range=(1, 3),
        skill_weight=0.35, 
        use_category_filter=True
    )
    
    # Train the engine (assuming 'df' from previous step is loaded)
    if 'df' in globals() and 'Cleaned_Resume' in df.columns:
        if 'Category' in df.columns:
            unique_categories = df['Category'].unique()
            logger.info(f"Available categories in dataset: {list(unique_categories)}")
        
        matcher.fit(
            resumes=df['Cleaned_Resume'],
            categories=df['Category'] if 'Category' in df.columns else None
        )
        
        # --- TEST 1: HR Role (using proper category filter) ---
        logger.info("\n" + "="*60)
        logger.info("TEST 1: Ranking HR candidates against HR job description")
        logger.info("="*60)
        
        hr_matches = matcher.rank_candidates(
            job_description=HR_JOB_DESCRIPTION,
            df_resumes=df,
            job_category='HR',  # CORRECT: Use 'HR' as seen in dataset
            top_n=10,
            min_score_threshold=15.0
        )
        
        print("\n" + "="*80)
        print("🏆 TOP HR CANDIDATES (Filtered by Category: HR) 🏆")
        print("="*80)
        display_results(hr_matches, show_details=True)
        
        # --- TEST 2: Technical Role (no category filter) ---
        logger.info("\n" + "="*60)
        logger.info("TEST 2: Ranking ALL candidates against Data Scientist role")
        logger.info("="*60)
        
        ds_matches = matcher.rank_candidates(
            job_description=DS_JOB_DESCRIPTION,
            df_resumes=df,
            job_category=None,  # No filter - look across all categories
            top_n=10,
            min_score_threshold=15.0
        )
        
        print("\n" + "="*80)
        print("🏆 TOP DATA SCIENTIST CANDIDATES (All Categories) 🏆")
        print("="*80)
        display_results(ds_matches, show_details=True)
        
        # --- TEST 3: Compare category distribution for a role ---
        if len(ds_matches) > 0:
            print("\n" + "="*80)
            print("📊 CATEGORY DISTRIBUTION FOR TOP DATA SCIENTIST MATCHES")
            print("="*80)
            category_counts = ds_matches['Category'].value_counts()
            print(category_counts.to_string())
        
        # --- Optional: Save model ---
        # matcher.save_model("hr_ats_matcher")
        
    else:
        logger.error("DataFrame 'df' with 'Cleaned_Resume' column not found. Please run Step 2 first.")

2026-04-01 23:42:31,561 - INFO - Initializing Enhanced ATS Matcher...
2026-04-01 23:42:31,626 - INFO - Available categories in dataset: ['HR', 'DESIGNER', 'INFORMATION-TECHNOLOGY', 'TEACHER', 'ADVOCATE', 'BUSINESS-DEVELOPMENT', 'HEALTHCARE', 'FITNESS', 'AGRICULTURE', 'BPO', 'SALES', 'CONSULTANT', 'DIGITAL-MEDIA', 'AUTOMOBILE', 'CHEF', 'FINANCE', 'APPAREL', 'ENGINEERING', 'ACCOUNTANT', 'CONSTRUCTION', 'PUBLIC-RELATIONS', 'BANKING', 'ARTS', 'AVIATION']
2026-04-01 23:42:31,628 - INFO - ==================================================
2026-04-01 23:42:31,629 - INFO - Training ATS Matcher
2026-04-01 23:42:31,630 - INFO - ==================================================
2026-04-01 23:42:31,787 - INFO - Converting 2483 resumes to TF-IDF vectors...
2026-04-01 23:42:46,469 - INFO - Vocabulary size: 8000 terms
2026-04-01 23:42:46,473 - INFO - Extracting skills from resumes...
2026-04-01 23:42:57,426 - INFO - Built category mapping for 24 categories                                            


🏆 TOP HR CANDIDATES (Filtered by Category: HR) 🏆

🏆 TOP CANDIDATE RANKINGS 🏆
 Rank Category Combined_Score Similarity_Score Skill_Score                                                                                                     Matched_Skills     Missing_Skills
    1       HR          48.8%            28.8%       85.7%                     training, performance management, hr compliance, payroll, recruitment, benefits administration employee relations
    2       HR          48.5%            28.5%       85.7%                     training, performance management, hr compliance, payroll, recruitment, benefits administration employee relations
    3       HR          46.0%            16.9%      100.0% training, performance management, employee relations, hr compliance, payroll, recruitment, benefits administration                   
    4       HR          45.9%            24.4%       85.7%                     training, performance management, hr compliance, payroll, recruitment, 

In [3]:
# ==========================================
# STEP 4: RECRUITER SUMMARY CARD (FINAL POLISH)
# ==========================================

import pandas as pd
import logging
from typing import Optional, Union, Dict, Any
from dataclasses import dataclass
from pathlib import Path

logger = logging.getLogger(__name__)


@dataclass
class RecommendationConfig:
    """Configuration for recruiter recommendations."""
    highly_recommended_threshold: float = 40.0
    potential_match_threshold: float = 20.0
    low_match_threshold: float = 0.0


class RecruiterSummaryGenerator:
    """
    Generates human-readable recruiter summary cards from candidate match data.
    Handles various edge cases and provides configurable thresholds.
    """
    
    def __init__(self, config: Optional[RecommendationConfig] = None):
        """
        Initialize the summary generator with optional custom thresholds.
        
        Args:
            config: RecommendationConfig with custom thresholds
        """
        self.config = config or RecommendationConfig()
        
    def _extract_score(self, score_value: Union[str, float, int]) -> float:
        """
        Safely extract numeric score from various input types.
        
        Args:
            score_value: Score could be string with '%', float, or int
            
        Returns:
            Float score value (0-100)
        """
        if isinstance(score_value, (int, float)):
            return float(score_value)
        
        if isinstance(score_value, str):
            # Remove '%' and any whitespace
            cleaned = score_value.replace('%', '').strip()
            try:
                return float(cleaned)
            except ValueError:
                logger.warning(f"Could not parse score: {score_value}")
                return 0.0
        
        return 0.0
    
    def _format_skills(self, skills: Union[str, list, None], max_skills: int = 15) -> str:
        """
        Format skills string for display, handling various input types.
        
        Args:
            skills: Comma-separated string, list, or None
            max_skills: Maximum number of skills to display
            
        Returns:
            Formatted skills string
        """
        if not skills:
            return "None"
        
        # Convert to list if string
        if isinstance(skills, str):
            if skills.lower() == 'none' or not skills.strip():
                return "None"
            skill_list = [s.strip() for s in skills.split(',') if s.strip()]
        elif isinstance(skills, list):
            skill_list = skills
        else:
            return "None"
        
        if not skill_list:
            return "None"
        
        # Limit and format
        if len(skill_list) > max_skills:
            formatted = ', '.join(skill_list[:max_skills])
            return f"{formatted} (+{len(skill_list) - max_skills} more)"
        
        return ', '.join(skill_list)
    
    def _get_recommendation(self, score: float, has_skills: bool = True) -> Dict[str, str]:
        """
        Generate recommendation based on score and skill presence.
        
        Args:
            score: Match score (0-100)
            has_skills: Whether candidate has any matching skills
            
        Returns:
            Dictionary with recommendation level and message
        """
        if score >= self.config.highly_recommended_threshold:
            return {
                'level': 'HIGHLY RECOMMENDED',
                'icon': '⭐',
                'message': 'Fast-track this candidate for an interview. Strong alignment with requirements.'
            }
        elif score >= self.config.potential_match_threshold:
            if has_skills:
                return {
                    'level': 'POTENTIAL MATCH',
                    'icon': '🔍',
                    'message': 'Worth a manual review. Candidate has some matching skills but may need additional screening.'
                }
            else:
                return {
                    'level': 'REVIEW SUGGESTED',
                    'icon': '📋',
                    'message': 'Moderate text similarity but no core skills detected. Consider manual resume review.'
                }
        else:
            if has_skills:
                return {
                    'level': 'SKILLED MISMATCH',
                    'icon': '⚠️',
                    'message': 'Low overall match despite some skill overlap. Role fit is questionable.'
                }
            else:
                return {
                    'level': 'LOW MATCH',
                    'icon': '❌',
                    'message': 'Does not align well with core requirements. Consider lower priority for review.'
                }
    
    def generate_summary(
        self,
        candidate_row: Union[pd.Series, Dict[str, Any]],
        job_title: str = "the role",
        show_technical_details: bool = True
    ) -> str:
        """
        Generate a human-readable recruiter summary card.
        
        Args:
            candidate_row: DataFrame row or dict containing candidate match data
            job_title: Title of the job being screened for
            show_technical_details: Whether to show TF-IDF similarity score
            
        Returns:
            Formatted summary string
        """
        # Handle dict input (for batch processing)
        if isinstance(candidate_row, dict):
            candidate_row = pd.Series(candidate_row)
        
        # Safely extract values with fallbacks
        rank = candidate_row.get('Rank', 'N/A')
        combined_score = candidate_row.get('Combined_Score', 0)
        similarity_score = candidate_row.get('Similarity_Score', 0)
        category = candidate_row.get('Category', 'Unknown')
        matched_skills = candidate_row.get('Matched_Skills', '')
        missing_skills = candidate_row.get('Missing_Skills', '')
        
        # Extract numeric score
        score_float = self._extract_score(combined_score)
        
        # Format skills
        matched_formatted = self._format_skills(matched_skills, max_skills=12)
        missing_formatted = self._format_skills(missing_skills, max_skills=8)
        
        # Get recommendation
        has_skills = matched_formatted != "None"
        rec = self._get_recommendation(score_float, has_skills)
        
        # Build summary
        summary_lines = []
        summary_lines.append("=" * 70)
        summary_lines.append(f"📄 RECRUITER SUMMARY CARD: CANDIDATE RANK #{rank}")
        summary_lines.append("=" * 70)
        
        # Top-Level Match
        summary_lines.append(f"\n🎯 OVERALL FIT: {combined_score}% Match for {job_title}")
        if show_technical_details:
            summary_lines.append(f"   └─ Technical Similarity: {similarity_score}%")
        summary_lines.append(f"📁 FOUND IN CATEGORY: {category}")
        
        # Skill Breakdown
        summary_lines.append("\n✅ SKILLS THEY HAVE (Matches):")
        summary_lines.append(f"   {matched_formatted}")
        
        summary_lines.append("\n❌ SKILLS THEY ARE MISSING (Gaps):")
        summary_lines.append(f"   {missing_formatted}")
        
        # AI Recommendation
        summary_lines.append(f"\n💡 AI RECOMMENDATION: {rec['icon']} {rec['level']}")
        summary_lines.append(f"   {rec['message']}")
        
        # Optional: Add interview focus areas if skills are missing
        if missing_formatted != "None" and score_float >= self.config.potential_match_threshold:
            summary_lines.append("\n🎤 INTERVIEW FOCUS AREAS:")
            summary_lines.append(f"   • Probe on missing skills: {missing_formatted}")
            summary_lines.append("   • Assess transferable experience and learning agility")
        
        summary_lines.append("\n" + "=" * 70)
        
        return '\n'.join(summary_lines)
    
    def generate_batch_summary(
        self,
        results_df: pd.DataFrame,
        job_title: str = "the role",
        output_file: Optional[Path] = None
    ) -> str:
        """
        Generate summaries for top candidates and optionally save to file.
        
        Args:
            results_df: DataFrame with ranking results
            job_title: Title of the job being screened for
            output_file: Optional file path to save summaries
            
        Returns:
            Combined summary string for all candidates
        """
        if results_df is None or len(results_df) == 0:
            logger.warning("No results to summarize")
            return ""
        
        all_summaries = []
        
        for idx, row in results_df.iterrows():
            summary = self.generate_summary(row, job_title, show_technical_details=True)
            all_summaries.append(summary)
            all_summaries.append("\n")  # Add spacing between candidates
        
        combined_summary = '\n'.join(all_summaries)
        
        # Save to file if requested
        if output_file:
            output_file = Path(output_file)
            output_file.parent.mkdir(parents=True, exist_ok=True)
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(combined_summary)
            logger.info(f"Summaries saved to {output_file}")
        
        return combined_summary
    
    def print_summary(self, candidate_row: Union[pd.Series, Dict[str, Any]], job_title: str = "the role"):
        """Print a formatted summary card."""
        summary = self.generate_summary(candidate_row, job_title)
        print(summary)


def generate_simple_summary(candidate_row: pd.Series, job_title: str = "the role") -> str:
    """
    Simple standalone function for quick summary generation.
    For production, use RecruiterSummaryGenerator class.
    
    Args:
        candidate_row: DataFrame row containing candidate match data
        job_title: Title of the job being screened for
        
    Returns:
        Formatted summary string
    """
    generator = RecruiterSummaryGenerator()
    return generator.generate_summary(candidate_row, job_title)


# ==========================================
# TEST THE SUMMARY CARD
# ==========================================

if __name__ == "__main__":
    
    # Check if we have results from Step 3
    if 'ds_matches' in locals() and ds_matches is not None and not ds_matches.empty:
        
        # Initialize the summary generator
        summary_gen = RecruiterSummaryGenerator(
            config=RecommendationConfig(
                highly_recommended_threshold=40.0,
                potential_match_threshold=20.0
            )
        )
        
        # TEST 1: Generate summary for top candidate
        print("\n" + "="*70)
        print("TEST 1: Summary Card for Top Candidate")
        print("="*70)
        
        top_candidate = ds_matches.iloc[0]
        summary_gen.print_summary(top_candidate, job_title="Data Scientist")
        
        # TEST 2: Generate summary for second candidate (if exists)
        if len(ds_matches) > 1:
            print("\n")
            print("="*70)
            print("TEST 2: Summary Card for Second Candidate")
            print("="*70)
            
            second_candidate = ds_matches.iloc[1]
            summary_gen.print_summary(second_candidate, job_title="Data Scientist")
        
        # TEST 3: Generate batch summaries and save to file
        print("\n" + "="*70)
        print("TEST 3: Batch Summaries for Top 5 Candidates")
        print("="*70)
        
        top_5 = ds_matches.head(5)
        
        # Save to file
        summary_gen.generate_batch_summary(
            results_df=top_5,
            job_title="Data Scientist",
            output_file=Path("recruiter_summaries.txt")
        )
        
        print("\n✅ Batch summaries saved to 'recruiter_summaries.txt'")
        
        # TEST 4: Test with custom thresholds
        print("\n" + "="*70)
        print("TEST 4: Summary with Stricter Thresholds")
        print("="*70)
        
        strict_generator = RecruiterSummaryGenerator(
            config=RecommendationConfig(
                highly_recommended_threshold=50.0,
                potential_match_threshold=30.0
            )
        )
        strict_generator.print_summary(top_candidate, job_title="Data Scientist")
        
    else:
        print("⚠️ No results found. Please run Step 3 first.")
        print("Make sure 'ds_matches' DataFrame is available.")

2026-04-01 23:48:48,584 - INFO - Summaries saved to recruiter_summaries.txt



TEST 1: Summary Card for Top Candidate
📄 RECRUITER SUMMARY CARD: CANDIDATE RANK #1

🎯 OVERALL FIT: 33.71% Match for Data Scientist
   └─ Technical Similarity: 15.96%
📁 FOUND IN CATEGORY: ENGINEERING

✅ SKILLS THEY HAVE (Matches):
   python, machine learning, statistics, data science

❌ SKILLS THEY ARE MISSING (Gaps):
   data analysis, pandas

💡 AI RECOMMENDATION: 🔍 POTENTIAL MATCH
   Worth a manual review. Candidate has some matching skills but may need additional screening.

🎤 INTERVIEW FOCUS AREAS:
   • Probe on missing skills: data analysis, pandas
   • Assess transferable experience and learning agility



TEST 2: Summary Card for Second Candidate
📄 RECRUITER SUMMARY CARD: CANDIDATE RANK #2

🎯 OVERALL FIT: 30.91% Match for Data Scientist
   └─ Technical Similarity: 20.62%
📁 FOUND IN CATEGORY: CONSULTANT

✅ SKILLS THEY HAVE (Matches):
   python, machine learning, data science

❌ SKILLS THEY ARE MISSING (Gaps):
   data analysis, pandas, statistics

💡 AI RECOMMENDATION: 🔍 POTENTIAL M